# Load libraries and enviornment variables

In [1]:
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate, FewShotChatMessagePromptTemplate
import os
from dotenv import dotenv_values, load_dotenv
import json
from typing import List, Literal
from typing_extensions import Annotated
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain.agents import tool
from langchain.tools.render import format_tool_to_openai_function
from langchain_core.runnables import RunnableLambda, RunnableMap, RunnablePassthrough, RunnableParallel
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser
from langchain.output_parsers import JsonOutputToolsParser
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser
from pydantic import BaseModel, Field, field_validator, model_validator
import requests
from langchain.schema import AIMessage, SystemMessage, HumanMessage
from langchain.callbacks import get_openai_callback
from concurrent.futures import ThreadPoolExecutor
from langchain.prompts import MessagesPlaceholder
from langchain.output_parsers import PydanticOutputParser
from langchain.agents.format_scratchpad import format_to_openai_functions
import time
import openpyxl
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import re
import ast

In [2]:
# this module contains all functions and classes used in this notebook
from functions_opensource import *

## Load PECOS criteria from a .yaml file

In [3]:
import yaml

# Load YAML file - use template to format pecos criteria
pecos_filename = "pecos_criteria.yaml"
with open(pecos_filename, "r") as file:
    config = yaml.safe_load(file)

# Access variables from YAML
pecos_population = config["population"]
pecos_exposure = config["exposure"] 
pecos_comparison = config["comparison"]
pecos_outcome = config["outcome"]
pecos_study_type = config["study_type"]     

## Load few-shot examples

In [4]:
# function to load examples from text files with python structures
def load_examples(filename):
    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()

    # Remove BOM if present
    text = text.lstrip("\ufeff")
    return ast.literal_eval(text)

# Load examples from text file, use the provided template for formatting. Use 4 examples
examples_filename = "examples.txt"
examples = load_examples(examples_filename) 

# Define model

In [5]:
llm = dotenv_values("config.txt")["OLLAMA_MODEL"]

In [6]:
# Load the Ollama endpoint from .env
load_dotenv()
ollama_base_url = os.getenv("OLLAMA_BASE_URL")

In [9]:
model = ChatOllama(
    base_url=ollama_base_url,
    model = llm,
    temperature = 0.0, #maximize determinism
    top_k=1, #maximize determinism
    num_ctx=8192, # this is the context window for the model. Older Ollama version had 2048 tokens as default, as of June 2026, they default to the maximum size. Previously, the context window seems to have been determined differently than with today's Ollama version. Higher num_ctx increase memory usage and may slow down inference.
#   top_p=0, --- IGNORE ---
    num_predict = 512, # this shouldn't be exceeded even with reasoning in the output 
    seed=123   # for reproducibility
)

## fewshot_large_noreason – Few-shot - no reason workflow with "large" model (Fs-NR-L)

In [10]:
system_noreason_fewshot_prompt = f"""You are conducting literature screening for systematic reviews and are helpful and thorough. Your task is to evaluate research studies and determine whether they should be included into the review stage. 
To determine eligibility of a study for inclusion, each of the following five study features has to be scrutinized for the following criteria:

*Study population*: {pecos_population}

*Exposure*: {pecos_exposure}

Comparator or control condition of the *Comparison*: {pecos_comparison}

*Outcome*: {pecos_outcome}

*Study Type*: {pecos_study_type}

After reading the title and abstract of a study, you will decide for each of the five study features if they meet the eligibility criteria. Apply thorough reasoning before you respond with your decision. You will use the ValidateForInclusionLight tool and fill out the key 'decision' for each element using 'True' or 'False', or 'NotEvaluable', if you are not sure whether to include or exclude the study. You will be shown some examples with explanations of the reasoning.
"""

In [11]:
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{response}"),
    ]
)

few_shot_message = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

In [12]:
noreason_fewshot_prompt = ChatPromptTemplate.from_messages([
    ("system", system_noreason_fewshot_prompt),
    few_shot_message,
    ("human", """
    Now, it is your turn to evaluate the following study text: {input} \n You MUST call and use the ValidateForInclusionLight tool to format your response. Do NOT return unstructured text incompatible with json. \n"""),
])

In [13]:
validateinclusion_noreason_function = [convert_to_openai_function(ValidateForInclusionLight)]

In [14]:
zero_shot_noreason_large_model = model.bind_tools(tools=validateinclusion_noreason_function)

In [15]:
parse_zeroshot_noreason = RunnableLambda(
    get_message_zeroshot_noreason
)

In [16]:
few_shot_large_noreason_chain = noreason_fewshot_prompt | zero_shot_noreason_large_model | parse_zeroshot_noreason | OpenAIFunctionsAgentOutputParser()

few_shot_large_noreason_workflow = few_shot_large_noreason_chain | format_zeroshot_noreason_results

# Load literature

## Option A: Read from Excel file

In [17]:
# Provide the path to the SR records Excel file.
# The file should have a Title and Abstract column.

path_SR_abs = "abs-screen.xlsx"

df = pd.read_excel(path_SR_abs)

In [18]:
df

,Title,Abstract
0,Magnetic field variability and cognitive perfo...,OBJECTIVE: To explore whether ambient magnetic...
1,Microbial adaptation to synthetic gravity in o...,OBJECTIVE: To assess whether microbial communi...
2,Effects of algorithmic soundscapes on plant gr...,OBJECTIVE: To determine whether exposure to al...


In [19]:
# get read of any "new lines" or "tabs"
df.replace({'\n': ' ', '\r': ' ', '\t': ' '}, regex=True, inplace=True)

# Create columns with explicit labels
df['Title_expl']= "Title: " + df['Title']
df['Abstract_expl']= "Abstract: " + df['Abstract']

In [20]:
#concatenate Title and Abstract into a single column, into a new data frame
data = pd.DataFrame()
data['Study_content'] = df['Title_expl'].astype(str).str.cat([df['Abstract_expl'].astype(str)], sep='. ')

In [21]:
data

,Study_content
0,Title: Magnetic field variability and cognitiv...
1,Title: Microbial adaptation to synthetic gravi...
2,Title: Effects of algorithmic soundscapes on p...


In [22]:
# generate a TiAb dictionary
abstracts_dict = [{"input": abstract} for abstract in data.Study_content.to_list()]

## Option B: Read from .ris file

In [23]:
# Provide the path to the SR records ris file.
# The file should have a TI and an AB header.

path_SR_abs = "abs-screen.ris"

df = read_ris(path_SR_abs)

In [24]:
df

,AB,AN,DA,DP,ET,IS,J2,KW,LA,LB,N1,PY,SN,SP,ST,T2,TI,VL,ID
0,OBJECTIVE: To explore whether ambient magnetic...,1000001,Jan,Nlm,2031/01/10,1,Journal of speculative computational science,Magnetic Fields Neural Networks Simulation Cog...,eng,fulltext,SSC-2030-01,2031,0000-0001 (Print) 0000-0001,1-5,Magnetic field variability and cognitive perfo...,J Spec Comput Sci,Magnetic field variability and cognitive perfo...,12,50001
1,OBJECTIVE: To assess whether microbial communi...,1000002,Apr,Nlm,2042/04/03,2,International journal of astro-biology experim...,Microbiology Gravity Simulation Evolution Bior...,eng,fulltext,ABE-2041-07,2042,0000-0002 (Print) 0000-0002,77-82,Microbial adaptation to synthetic gravity in o...,Int J Astro-Biol Exp,Microbial adaptation to synthetic gravity in o...,5,50002
2,OBJECTIVE: To determine whether exposure to al...,1000003,Oct,Nlm,2037/10/21,4,Journal of experimental bioacoustics,Bioacoustics Plant Growth Algorithms Sound Exp...,eng,fulltext,JEB-2037-19,2037,0000-0003 (Print) 0000-0003,210-214,Effects of algorithmic soundscapes on plant gr...,J Exp Bioacoust,Effects of algorithmic soundscapes on plant gr...,9,50003


In [25]:
# get read of any "new lines" or "tabs"
df.replace({'\n': ' ', '\r': ' ', '\t': ' '}, regex=True, inplace=True)

# Create columns with explicit labels
df['Title_expl']= "Title: " + df['TI']
df['Abstract_expl']= "Abstract: " + df['AB']

In [26]:
#concatenate Title and Abstract into a single column, into a new data frame
data = pd.DataFrame()
data['Study_content'] = df['Title_expl'].astype(str).str.cat([df['Abstract_expl'].astype(str)], sep='. ')

In [27]:
data

,Study_content
0,Title: Magnetic field variability and cognitiv...
1,Title: Microbial adaptation to synthetic gravi...
2,Title: Effects of algorithmic soundscapes on p...


In [28]:
# generate a TiAb dictionary
abstracts_dict = [{"input": abstract} for abstract in data.Study_content.to_list()]

# Screen literature

In [29]:
# create a directory to save pickled results

savedir_pickle = f'res-SR-pickled'
if not os.path.exists(savedir_pickle):
    os.mkdir(savedir_pickle)

In [ ]:
# Run few-shot large-noreason workflow in batch mode, save results every 200 records
# Allow for 2 min pauses to avoid overloading the system. Adjust as needed.
time_to_sleep = 120  # time to sleep in seconds

# Computation time is also tracked

# name of the batch results
name_output = "fewshot_large_noreason"

batch_results_fewshot_large_noreason = [None] * len(abstracts_dict)

start_time = datetime.now()

for i, abstract in enumerate(abstracts_dict):
    print(i)
    
    result = few_shot_large_noreason_workflow.invoke(abstract)
    batch_results_fewshot_large_noreason[i] =  result

    if i in list(range(199,len(abstracts_dict),200)):
        save_batch(batch_results_fewshot_large_noreason, 
                   f'batch_results_{name_output}', 
                   savedir_pickle)
        time.sleep(time_to_sleep)
    time.sleep(0.2)

end_time = datetime.now()

comptime_fewshot_large_noreason = end_time - start_time
comptime_fewshot_large_noreason = comptime_fewshot_large_noreason.total_seconds() / 60

save_batch(batch_results_fewshot_large_noreason, 
           f'batch_results_{name_output}', 
           savedir_pickle)

print(f"Computation time for few-shot large-noreason workflow: {comptime_fewshot_large_noreason} minutes.")

In [24]:
batch_results_fewshot_large_noreason[0:3]

[{'evaluations': {'population_evaluation': {'decision': 'NotEvaluable',
    'reason': 'No reason requested.'},
   'exposure_evaluation': {'decision': 'True',
    'reason': 'No reason requested.'},
   'comparison_evaluation': {'decision': 'True',
    'reason': 'No reason requested.'},
   'outcome_evaluation': {'decision': 'True',
    'reason': 'No reason requested.'},
   'study_type_evaluation': {'decision': 'False',
    'reason': 'No reason requested.'}},
  'summary': {'True': 3, 'False': 1, 'NotEvaluable': 1},
  'PECOS': {}},
 {'evaluations': {'population_evaluation': {'decision': 'True',
    'reason': 'No reason requested.'},
   'exposure_evaluation': {'decision': 'True',
    'reason': 'No reason requested.'},
   'comparison_evaluation': {'decision': 'True',
    'reason': 'No reason requested.'},
   'outcome_evaluation': {'decision': 'True',
    'reason': 'No reason requested.'},
   'study_type_evaluation': {'decision': 'False',
    'reason': 'No reason requested.'}},
  'summary': {'

# Define decision rule

## Decision rule 6 - sensitive, allow manual disambiguitation
\>= 3/5 PECOS fulfilled -> include  
<3/5 PECOS fulfilled and >= 2/5 PECOS not fulfilled -> exclude  
if none of the above applies and >= 2/5 PECOS not evaluable -> tag for manual check

In [25]:
def alg6(batch_results):
    decision_restults_alg6 = []
    for i in [entry['summary'] for entry in batch_results]:
        if i['True'] >= 3:
            decision_restults_alg6.append('include')
        elif i['True'] < 3 and i['False'] >= 2:
            decision_restults_alg6.append('exclude')
        elif i['NotEvaluable'] >= 2:
            decision_restults_alg6.append('MANUAL_CHECK')
        else:
            decision_restults_alg6.append('exclude')
    return decision_restults_alg6

In [26]:
alg_functions = {
    'alg6_sens_man': alg6
}

def decision_alg(batch_results):
    results = []
    for func_name, func in alg_functions.items():
        result = {
            'algorithm': func_name,
            'predictions': func(batch_results)
        }
        results.append(result)
    return results

# Derive decisions from batch results and export labels and evaluations

## Load results

In [27]:
savedir_pickle = f'res-SR-pickled' # set to your results folder
name_output = "fewshot_large_noreason" # name of the workflow
file_path = f'{savedir_pickle}/batch_results_{name_output}.pkl'

with open(file_path, 'rb') as f:
    batch_results_fewshot_mid = pickle.load(f)


## Process results

In [28]:
# directory to save results in
savedir_res = 'res-SR'

if not os.path.exists(savedir_res):
    os.mkdir(savedir_res)

In [29]:
# Generate decisions using algorithms
decisions = decision_alg(batch_results_fewshot_large_noreason)

In [30]:
# Create expanded DataFrame from original data
expanded_df = data.copy()

name_output = "fewshot_large_noreason" # name of the workflow
results = {} # dictionary to hold results
results[name_output] = {
    'batch_results': batch_results_fewshot_large_noreason,
    'decisions': decisions
}

# Add evaluations columns
expanded_df[f'{name_output}_evaluations'] = [item['evaluations'] if 'evaluations' in item else None 
                                        for item in results[name_output]['batch_results']]
# Add calls columns
expanded_df[f'{name_output}_calls'] = [item['summary'] if 'summary' in item else None 
                                    for item in results[name_output]['batch_results']]

# Add decision columns from each decision rule
decisions_df = pd.DataFrame({
r['algorithm']: r['predictions'] 
for r in results[name_output]['decisions']
})

for col in decisions_df.columns:
    expanded_df[f'{name_output}_{col}'] = decisions_df[col]

# Export everything
expanded_df.to_excel(f'{savedir_res}/complete_results.xlsx', index=False)

# Add decision label back to original import

## Option A: Export back to excel

In [31]:
df['Label'] = decisions[0]['predictions']

In [32]:
df

,Title,Abstract,Title_expl,Abstract_expl,Label
0,Magnetic field variability and cognitive perfo...,OBJECTIVE: To explore whether ambient magnetic...,Title: Magnetic field variability and cognitiv...,Abstract: OBJECTIVE: To explore whether ambien...,include
1,Microbial adaptation to synthetic gravity in o...,OBJECTIVE: To assess whether microbial communi...,Title: Microbial adaptation to synthetic gravi...,Abstract: OBJECTIVE: To assess whether microbi...,include
2,Effects of algorithmic soundscapes on plant gr...,OBJECTIVE: To determine whether exposure to al...,Title: Effects of algorithmic soundscapes on p...,Abstract: OBJECTIVE: To determine whether expo...,include


In [ ]:
df.drop(columns=['Title_expl', 'Abstract_expl'])
df

In [33]:
df.to_excel(f'{savedir_res}/abs-screen_labeled.xlsx', index=False)

## Option B: Export back to ris

In [40]:
df['LB'] = decisions[0]['predictions']

In [41]:
df.drop(columns=['Title_expl', 'Abstract_expl'])
df

,AB,AN,DA,DP,ET,IS,J2,KW,LA,LB,...,PY,SN,SP,ST,T2,TI,VL,ID,Title_expl,Abstract_expl
0,OBJECTIVE: To explore whether ambient magnetic...,1000001,Jan,Nlm,2031/01/10,1,Journal of speculative computational science,Magnetic Fields Neural Networks Simulation Cog...,eng,include,...,2031,0000-0001 (Print) 0000-0001,1-5,Magnetic field variability and cognitive perfo...,J Spec Comput Sci,Magnetic field variability and cognitive perfo...,12,50001,Title: Magnetic field variability and cognitiv...,Abstract: OBJECTIVE: To explore whether ambien...
1,OBJECTIVE: To assess whether microbial communi...,1000002,Apr,Nlm,2042/04/03,2,International journal of astro-biology experim...,Microbiology Gravity Simulation Evolution Bior...,eng,include,...,2042,0000-0002 (Print) 0000-0002,77-82,Microbial adaptation to synthetic gravity in o...,Int J Astro-Biol Exp,Microbial adaptation to synthetic gravity in o...,5,50002,Title: Microbial adaptation to synthetic gravi...,Abstract: OBJECTIVE: To assess whether microbi...
2,OBJECTIVE: To determine whether exposure to al...,1000003,Oct,Nlm,2037/10/21,4,Journal of experimental bioacoustics,Bioacoustics Plant Growth Algorithms Sound Exp...,eng,include,...,2037,0000-0003 (Print) 0000-0003,210-214,Effects of algorithmic soundscapes on plant gr...,J Exp Bioacoust,Effects of algorithmic soundscapes on plant gr...,9,50003,Title: Effects of algorithmic soundscapes on p...,Abstract: OBJECTIVE: To determine whether expo...


In [42]:
write_ris(df, f'{savedir_res}/abs-screen_labeled.ris')